In [ ]:
# 1. INITIALIZATION
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, when, lit
import snowflake.snowpark.functions as F

session = get_active_session()

def run_retail_scorecard(session):
    # 2. LOAD SOURCE DATA
    # Ensure the table name matches what you created in Snowflake
    df = session.table("RETAIL_CREDIT_RISK.CREDIT_SCHEMA.RETAIL_LOAN_APPLICATIONS")

    # 3. SCORECARD LOGIC (PILLAR 1 & 2)
    # Calculate FOIR (Fixed Obligation to Income Ratio)
    # Higher EMI/Income ratio = Lower Score. 
    # Example: If EMI is 20% of income, FOIR_SCORE is 80.
    df_calc = df.with_column("FOIR_SCORE", 
        100 - ((col("EXISTING_EMI_TOTAL") / (col("ANNUAL_INCOME") / 12)) * 100))

    # Normalize CIBIL (Mapping 300-900 to 0-100)
    df_calc = df_calc.with_column("CIBIL_NORM", (col("CIBIL_SCORE") - 300) / 6)

    # Calculate BASE_RISK_SCORE (50% CIBIL, 30% FOIR, 20% Job Stability)
    # Assuming 15 years is 'Max Stability' for the 100-point scale
    df_calc = df_calc.with_column("RISK_SCORE", 
        (col("CIBIL_NORM") * 0.5) + 
        (col("FOIR_SCORE") * 0.3) + 
        ((col("YEARS_AT_CURRENT_JOB") / 15) * 100 * 0.2))

    # 4. FRAUD LOGIC (PILLAR 3)
    # Calculate Fraud Probability based on Velocity and Geo-mismatch
    df_calc = df_calc.with_column("FRAUD_PROBABILITY", 
        when(col("IP_ADDRESS_GEO_MATCH") == False, 0.4).otherwise(0) + 
        (col("APPLICATION_VELOCITY") * 0.05))

    # 5. VERDICT ENGINE (THE DECISION)
    # Rule 1: Knock-out (KO) for Identity Mismatch or High Fraud Prob
    # Rule 2: Approve if Risk Score > 75
    # Rule 3: Refer for Manual Review if 50-75
    df_final = df_calc.with_column("DECISION", 
        when((col("PAN_AADHAAR_MATCH") == False) | (col("KYC_STATUS") == 'Flagged'), lit("REJECT - FRAUD"))
        .when(col("FRAUD_PROBABILITY") > 0.6, lit("REJECT - HIGH VELOCITY"))
        .when(col("RISK_SCORE") > 75, lit("APPROVE"))
        .when(col("RISK_SCORE") > 50, lit("REFER - MANUAL"))
        .otherwise(lit("REJECT - CREDIT")))

    # 6. WRITE RESULTS TO SNOWFLAKE
    # Using overwrite to ensure we have a fresh table with all 100 rows populated
    df_final.write.mode("overwrite").save_as_table("RETAIL_CREDIT_RISK.CREDIT_SCHEMA.RETAIL_LOAN_RESULTS")
    
    return df_final

# EXECUTION
# Run the function and capture the result in the notebook scope
final_results_df = run_retail_scorecard(session)

# DISPLAY OUTPUT
print("Transformation Complete. Table 'RETAIL_LOAN_RESULTS' updated.")
final_results_df.select(
    "CUSTOMER_NAME", 
    "CIBIL_SCORE", 
    "RISK_SCORE", 
    "FRAUD_PROBABILITY", 
    "DECISION"
).limit(100).to_pandas()